In [1]:
#r "ServerThreadTests\bin\Debug\net10.0\ServerThreadTests.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class TestCommand : ILongCommand
{
    private readonly int _id;
    private int _counter;
    private readonly int _maxCalls;
    
    public bool IsCompleted => _counter >= _maxCalls;
    
    public TestCommand(int id, int maxCalls = 3)
    {
        _id = id;
        _maxCalls = maxCalls;
        _counter = 0;
    }
    
    public void Execute()
    {
        _counter++;
        Console.WriteLine($"Поток {_id} вызов {_counter}");
    }
}

public class HeavyCommand : ILongCommand
{
    private readonly int _workload;
    private readonly int _maxCalls;
    private int _calls;
    
    public bool IsCompleted => _calls >= _maxCalls;
    
    public HeavyCommand(int workload = 100000, int maxCalls = 1)
    {
        _workload = workload;
        _maxCalls = maxCalls;
        _calls = 0;
    }
    
    public void Execute()
    {
        double x = 0;
        for (int i = 0; i < _workload; i++)
            x += Math.Sin(i) * Math.Cos(i);
        _calls++;
    }
}

var scheduler = new RoundRobinScheduler();
var serverThread = new ServerThread(scheduler);
serverThread.Start();

for (int i = 1; i <= 5; i++)
{
    scheduler.Add(new TestCommand(i, 3));
}

Thread.Sleep(100);

while (scheduler.HasCommand())
{
    Thread.Sleep(50);
}

serverThread.Add(new HardStopCommand(serverThread));
serverThread.Thread.Join();

Console.WriteLine("\nПоток остановлен командой HardStop.\n");

Console.WriteLine("Замеры производительности: Один поток / Round Robin\n");

var counts = new int[] { 1, 2, 4, 8, 16 };
var execPerCmd = 10;
var singleTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double singleTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(100000, execPerCmd);
            while (!cmd.IsCompleted)
                cmd.Execute();
        }
        sw.Stop();
        singleTotal += sw.Elapsed.TotalMilliseconds;
    }
    singleTimes[i] = singleTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var s = new RoundRobinScheduler();
        var st = new ServerThread(s);
        var sw = Stopwatch.StartNew();

        st.Start();
        for (int j = 0; j < n; j++)
        {
            s.Add(new HeavyCommand(100000, execPerCmd));
        }
        
        Thread.Sleep(100);

        while (s.HasCommand())
        {
            Thread.Sleep(10);
        }

        sw.Stop();
        st.Add(new HardStopCommand(st));
        st.Thread.Join();
        
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;

    Console.WriteLine($"Команд: {n,2} | Один поток: {singleTimes[i],8:F1} мс | Round Robin: {rrTimes[i],8:F1} мс");
}

var plot = new Plot();
plot.Title("Время выполнения: Один поток/Round Robin");
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

var xs = counts.Select(x => (double)x).ToArray();
plot.Add.Scatter(xs, singleTimes).Label = "Один поток";
plot.Add.Scatter(xs, rrTimes).Label = "Round Robin";
plot.ShowLegend();

var fn = "plot.png";
plot.SavePng(fn, 800, 600);
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

var reportPath = "report.txt";
var sb = new System.Text.StringBuilder();

sb.AppendLine($"ОТЧЁТ");
sb.AppendLine($"Дата: {DateTime.Now:dd.MM.yyyy}");
sb.AppendLine();
sb.AppendLine("1. TestCommand реализует ILongCommand.");
sb.AppendLine("   Выполняется за 3 вызова Execute, выводит счётчик.");
sb.AppendLine();
sb.AppendLine("2. 5 экземпляров TestCommand добавлены в RoundRobinScheduler,");
sb.AppendLine("   выполнены по 3 раза каждый с чередованием.");
sb.AppendLine();
sb.AppendLine("3. Поток остановлен HardStop после завершения всех команд.");
sb.AppendLine();
sb.AppendLine("4. Замеры (HeavyCommand, 100K итераций × 10 вызовов):");
for (int i = 0; i < counts.Length; i++)
    sb.AppendLine($"   Команд: {counts[i],2} | Один поток: {singleTimes[i],7:F1} мс | Round Robin: {rrTimes[i],7:F1} мс");
sb.AppendLine();
sb.AppendLine("5. Вывод: Round Robin выполняет длительные операции");
sb.AppendLine("   без монопольной блокировки потока, с возможностью");
sb.AppendLine("   прерывания, без существенных накладных расходов.");

File.WriteAllText(reportPath, sb.ToString());
Console.WriteLine($"\nОтчет сохранен: {reportPath}");
Console.WriteLine($"График сохранен: {fn}");

Installed Packages ScottPlot, 5.0.54

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.9\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3

Поток остановлен командой HardStop.

Замеры производительности: Один поток / Round Robin

Команд:  1 | Один поток:     24.1 мс | Round Robin:    106.7 мс
Команд:  2 | Один поток:     48.1 мс | Round Robin:    107.4 мс
Команд:  4 | Один поток:     97.7 мс | Round Robin:    113.0 мс
Команд:  8 | Один поток:    184.1 мс | Round Robin:    198.8 мс
Команд: 16 | Один поток:    379.1 мс | Round Robin:    392.8 мс



Отчет сохранен: report.txt
График сохранен: plot.png
